# Lab 01 · Faça a reta aprender — versão guiada
**Dia 1 · Módulo 02 · 25 minutos**

Você vai treinar uma reta para prever a nota de um aluno, medir seus erros, adicionar mais informações e verificar se aumentar a complexidade ajuda.

Execute as células em ordem. Complete somente os `___` das células TODO. As demais estão prontas. Use o Google Colab.

| Etapa | Tempo |
|---|---:|
| 1. Conhecer os dados | 3 min |
| 2. Fazer a reta aprender | 7 min |
| 3. Treinar e medir com scikit-learn | 5 min |
| 4. Adicionar mais features | 5 min |
| 5. Comparar graus por validação cruzada | 5 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (8, 4)

In [ ]:
# @title Carregar os dados do Lab 00 (execute; não é necessário editar)
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

def limpar(df):
    """A mesma limpeza do Lab 00."""
    df = df.drop_duplicates().copy()
    df['turno'] = df['turno'].str.strip().str.lower().replace({'manha': 'manhã'})
    df.loc[df['horas_estudo'] > 24, 'horas_estudo'] = np.nan
    return df.dropna(subset=['nota_final']).reset_index(drop=True)

import os
df = pd.read_csv('alunos_limpo.csv') if os.path.exists('alunos_limpo.csv') else limpar(gerar_dados())
print(df.shape)
df.head()

## 1 · Conhecer os dados (3 min)
Vamos prever `nota_final` usando `horas_estudo`. Para o primeiro modelo, retiramos alunos sem horas registradas. Usaremos **os mesmos alunos e a mesma divisão** nos modelos seguintes para compará-los de maneira justa. O teste não participa dos ajustes.

In [ ]:
d = df.dropna(subset=['horas_estudo', 'nota_final']).copy()
idx_tr, idx_te = train_test_split(d.index.to_numpy(), test_size=0.2, random_state=42)
x_tr = d.loc[idx_tr, 'horas_estudo'].to_numpy()
x_te = d.loc[idx_te, 'horas_estudo'].to_numpy()
y_tr = d.loc[idx_tr, 'nota_final'].to_numpy()
y_te = d.loc[idx_te, 'nota_final'].to_numpy()

print(f'Treino: {len(x_tr)} | Teste: {len(x_te)}')
plt.scatter(x_tr, y_tr, alpha=.5)
plt.xlabel('Horas de estudo'); plt.ylabel('Nota final'); plt.show()

**Observe:** a tendência é crescente? Uma reta consegue passar por todos os pontos?

## 2 · Fazer a reta aprender (7 min)
O modelo é $\hat y = wx+b$. O resíduo é $y-\hat y$ e a perda é a média dos resíduos ao quadrado (MSE).

Os gradientes já estão calculados na função abaixo. Eles dizem como a perda varia com cada parâmetro. Para diminuir a perda, subtraímos `alpha * gradiente` do parâmetro.

**TODO:** complete as três lacunas: previsão e atualização de `w` e `b`.

In [ ]:
def descida_gradiente(x, y, alpha=0.02, epocas=3000):
    w, b = 0.0, 0.0
    historico = []
    for _ in range(epocas):
        y_hat = ___  # TODO: previsão
        erro = y - y_hat
        historico.append((erro ** 2).mean())
        dw = -2 * (x * erro).mean()
        db = -2 * erro.mean()
        w = ___  # TODO: atualizar w
        b = ___  # TODO: atualizar b
    return w, b, historico

w, b, hist = descida_gradiente(x_tr, y_tr)
print(f'w = {w:.3f} | b = {b:.3f} | MSE final = {((y_tr - (w*x_tr+b))**2).mean():.3f}')
plt.plot(hist); plt.yscale('log')
plt.xlabel('Época'); plt.ylabel('MSE (treino)'); plt.show()

### Experimente a taxa de aprendizado
Aumentar `alpha` não garante aprender mais rápido. Execute e compare as curvas; a divergência é exibida até MSE = 1000 para facilitar a leitura.

In [ ]:
taxas = [0.001, 0.01, 0.023, 0.03]
for alpha in taxas:
    _, _, h = descida_gradiente(x_tr, y_tr, alpha=alpha, epocas=300)
    plt.plot(np.clip(h, None, 1e3), label=f'alpha = {alpha}')
plt.yscale('log'); plt.xlabel('Época'); plt.ylabel('MSE (treino)')
plt.legend(); plt.show()

**Observe:** qual taxa é lenta? Qual diverge? Por que aumentar o número de épocas não resolve uma taxa grande demais?

## 3 · Treinar e medir com scikit-learn (5 min)
Agora a biblioteca ajusta os parâmetros por mínimos quadrados. O objetivo é comparar com a reta anterior.

**TODO:** use `fit` para treinar e `predict` para prever. `reshape(-1, 1)` transforma o vetor em uma matriz com uma coluna.

In [ ]:
reg = LinearRegression()
reg.___(x_tr.reshape(-1, 1), y_tr)  # TODO: treinar
novos = np.array([2, 6, 10])
notas = reg.___(novos.reshape(-1, 1))  # TODO: prever
print('NumPy:', round(w, 3), round(b, 3))
print('Scikit-learn:', round(reg.coef_[0], 3), round(reg.intercept_, 3))
for horas, nota in zip(novos, notas):
    print(f'{horas} h → nota {nota:.2f}')
xs = np.linspace(0, 15, 100)
plt.scatter(x_tr, y_tr, alpha=.4)
plt.plot(xs, w * xs + b, 'C1', lw=3, label='Gradiente')
plt.plot(xs, reg.predict(xs.reshape(-1, 1)), 'k--', label='Scikit-learn')
plt.legend(); plt.xlabel('Horas'); plt.ylabel('Nota'); plt.show()

### Medir no teste
MAE: erro absoluto médio, em pontos de nota. MSE: erro quadrático médio, em pontos². R²: redução relativa do erro quadrático em relação a prever a média dos valores reais avaliados; pode ser negativo. Não é acurácia.

In [ ]:
def avaliar(real, previsto):
    return {'MAE': mean_absolute_error(real, previsto),
            'MSE': mean_squared_error(real, previsto),
            'R²': r2_score(real, previsto)}

pred = reg.predict(x_te.reshape(-1, 1))
resultado_simples = avaliar(y_te, pred)
pd.DataFrame([resultado_simples], index=['Só horas']).round(3)

**Observe:** o que significa um MAE de aproximadamente 0,8 ponto numa escala de 0 a 10?

## 4 · Adicionar mais features (5 min)
A mesma nota pode depender de horas, faltas, turno e outras informações. O Pipeline abaixo preenche valores faltantes, padroniza as features e treina a regressão. Imputação e escala são ajustadas somente no treino (e dentro de cada fold na validação cruzada).

As categorias de turno e monitoria já são conhecidas no Lab 00. Fixamos essas categorias antes do one-hot, removendo uma coluna de referência por categoria; não usamos `situacao`, pois ela é derivada da nota que queremos prever. A preparação está pronta.

In [ ]:
X = d.drop(columns=['nota_final', 'situacao']).copy()
X['turno'] = pd.Categorical(X['turno'], categories=['manhã', 'tarde', 'noite'])
X['monitoria'] = pd.Categorical(X['monitoria'], categories=['não', 'sim'])
X = pd.get_dummies(X, columns=['turno', 'monitoria'], drop_first=True, dtype=int)
X_tr, X_te = X.loc[idx_tr], X.loc[idx_te]

multipla = Pipeline([
    ('imputar', SimpleImputer(strategy='median')),
    ('padronizar', StandardScaler()),
    ('regressao', LinearRegression()),
])
multipla.fit(X_tr, y_tr)
resultado_multipla = avaliar(y_te, multipla.predict(X_te))
pd.DataFrame([resultado_simples, resultado_multipla], index=['Só horas', 'Várias features']).round(3)

**Observe:** compare o MAE dos dois modelos nos mesmos alunos. Adicionar informações garantiu uma melhora? Essas duas avaliações são descritivas, não uma seleção de modelo pelo teste.

## 5 · O modelo mais complexo é melhor? (5 min)
`PolynomialFeatures` transforma a entrada em potências de x. Um grau maior permite curvas mais flexíveis, mas pode aumentar o erro em dados novos.

Vamos comparar os graus **1, 2 e 10** pelo MAE médio de 5 folds, todos dentro do treino. O teste fica fora dessa escolha. A pontuação `neg_mean_absolute_error` vem negativa no scikit-learn; o sinal de menos a converte novamente em erro.

In [ ]:
graus = [1, 2, 10]
resultados = []
for grau in graus:
    poli = Pipeline([
        ('poli', PolynomialFeatures(degree=grau, include_bias=False)),
        ('padronizar', StandardScaler()),
        ('regressao', LinearRegression()),
    ])
    erros = -cross_val_score(poli, x_tr.reshape(-1, 1), y_tr,
                            cv=5, scoring='neg_mean_absolute_error')
    poli.fit(x_tr.reshape(-1, 1), y_tr)
    mae_treino = mean_absolute_error(y_tr, poli.predict(x_tr.reshape(-1, 1)))
    resultados.append({'grau': grau, 'MAE treino': mae_treino,
                       'MAE validação': erros.mean()})
comparacao = pd.DataFrame(resultados).set_index('grau')
display(comparacao.round(3))
comparacao.plot.bar(rot=0); plt.ylabel('MAE (pontos)'); plt.show()
print('Menor MAE de validação: grau', comparacao['MAE validação'].idxmin())

**Conclusão:** qual grau você escolheria pela validação? Um MAE de treino menor garante desempenho em dados novos? Neste exercício, não reutilizamos o teste para escolher graus.

## Para lembrar
Dados → previsão → erro → ajuste → avaliação → generalização.